# Deep Learning Text Generation
This notebook designs and implements a DL model capable of learning the underlying structure, grammar, and contextual dependencies of a given text corpus to generate coherent text sequences using **Vanilla SimpleRNN, LSTM, and GRU**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense

## 1. Ingest, Clean, and Tokenize the Custom Corpus

In [ ]:
# Custom paragraph corpus
text_corpus = """
Deep learning models, like recurrent neural networks, are particularly good at processing sequential data.
These models have revolutionized natural language processing and text generation.
By understanding the context and grammar of a corpus, they can produce highly coherent sentences.
Artificial intelligence is a fascinating field of computer science.
It aims to create systems capable of performing tasks that normally require human intelligence.
Such tasks include learning, reasoning, problem-solving, perception, and language understanding.
"""

# Clean the input text corpus string
text_corpus = text_corpus.lower().strip()
corpus = text_corpus.split('\n')
corpus = [line.strip() for line in corpus if line.strip()]

# Initialize a standard text tokenization instance to map words to integers
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1

print(f"Total unique words in corpus: {total_words}")

## 2. Reframe Word Index Tracking into N-Grams and Pad Sequences

In [ ]:
# Reframe word index tracking lists into progressive sliding-window combinations (n-grams)
input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Match vector dimensions using pad_sequences
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Create predictors and labels
X, y = input_sequences[:,:-1], input_sequences[:,-1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print(f"Max sequence length: {max_sequence_len}")
print(f"Predictors shape: {X.shape}, Labels shape: {y.shape}")

## 3. Build Three Separate Text Sequence Models

In [ ]:
# Customization Tasks:
embedding_dim = 128  # Upscaled embedding dimensions
hidden_units = 128   # Widened hidden layers (64 -> 128)
epochs = 200         # Expanded training to 200 epochs

def create_model(model_type):
    model = Sequential()
    model.add(Embedding(total_words, embedding_dim, input_length=max_sequence_len-1))
    
    if model_type == 'RNN':
        model.add(SimpleRNN(hidden_units))
    elif model_type == 'LSTM':
        model.add(LSTM(hidden_units))
    elif model_type == 'GRU':
        model.add(GRU(hidden_units))
        
    model.add(Dense(total_words, activation='softmax'))
    # Identical optimizer configuration
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

print("Building models...")
model_rnn = create_model('RNN')
model_lstm = create_model('LSTM')
model_gru = create_model('GRU')

## 4. Train Models Over 200 Epochs

In [ ]:
# Train all three variations using an identical optimizer configuration
print(f"Training Vanilla SimpleRNN for {epochs} epochs...")
history_rnn = model_rnn.fit(X, y, epochs=epochs, verbose=0)

print(f"Training LSTM for {epochs} epochs...")
history_lstm = model_lstm.fit(X, y, epochs=epochs, verbose=0)

print(f"Training GRU for {epochs} epochs...")
history_gru = model_gru.fit(X, y, epochs=epochs, verbose=0)
print("Training Complete!")

## 5. Map Optimization Trajectories (Training Curves)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(history_rnn.history['loss'], label='SimpleRNN Loss')
plt.plot(history_lstm.history['loss'], label='LSTM Loss')
plt.plot(history_gru.history['loss'], label='GRU Loss')
plt.title('Training Cross-Entropy Loss Reduction over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

## 6 & 7. Generate Text Using np.argmax Logic

In [ ]:
def generate_text(seed_text, next_words, model, max_sequence_len):
    output_text = seed_text
    for _ in range(next_words):
        # Tokenize the current string
        token_list = tokenizer.texts_to_sequences([output_text])[0]
        # Pad the sequence
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        # Predict the next word probabilities
        predicted_probs = model.predict(token_list, verbose=0)
        # Use np.argmax over next-word probability arrays
        predicted_word_index = np.argmax(predicted_probs, axis=-1)[0]
        
        # Map back to word
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_word_index:
                output_word = word
                break
        output_text += " " + output_word
    return output_text

# Adjust output to return 10 words per generation prompt
seed_phrase = "Artificial intelligence is"
words_to_generate = 10

print(f"--- Text Generation (Seed: '{seed_phrase}') ---\n")

text_rnn = generate_text(seed_phrase, words_to_generate, model_rnn, max_sequence_len)
print(f"Vanilla SimpleRNN:\n{text_rnn}\n")

text_lstm = generate_text(seed_phrase, words_to_generate, model_lstm, max_sequence_len)
print(f"LSTM:\n{text_lstm}\n")

text_gru = generate_text(seed_phrase, words_to_generate, model_gru, max_sequence_len)
print(f"GRU:\n{text_gru}")